<a href="https://www.kaggle.com/code/muhammadmubeenxt/flooding-experiment-9-march-13?scriptVersionId=303622183" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Libraries

In [2]:
# Run this cell first to install all required packages
!pip install pystac-client planetary-computer stackstac pyproj rioxarray dask odc-stac -q
!pip install rasterio python-dotenv requests segmentation-models-pytorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.3/64.3 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.0/159.0 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.5/208.5 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 35.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 5.0 MB/s eta 0:00:00


# Multiple imagery download

In [ ]:
import os
import sys
import time
import logging
import gc
import pandas as pd
import numpy as np
import pystac_client
import planetary_computer
import odc.stac
import rioxarray
import xarray as xr
from rasterio.enums import Resampling
from requests.exceptions import ReadTimeout

# Output to stdout to prevent environmental red-text highlighting
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("bulk_download.log"),
        logging.StreamHandler(sys.stdout)
    ]
)

# Canonical band order — strictly locked for U-Net multi-channel input
BAND_ORDER = ['SAR_CoPol', 'SAR_CrossPol', 'B02', 'B03', 'B04', 'B08', 'SCL']

def fetch_fusion_ready_imagery(lat, lon, radius_km, target_date, event_label, output_dir):
    target_dt     = pd.to_datetime(target_date, utc=True)
    degree_buffer = radius_km / 111.0
    bbox          = [lon - degree_buffer, lat - degree_buffer,
                     lon + degree_buffer, lat + degree_buffer]

    logging.info(f"Processing: {event_label} | Target: {target_date}")

    try:
        catalog = pystac_client.Client.open(
            "https://planetarycomputer.microsoft.com/api/stac/v1",
            modifier=planetary_computer.sign_inplace,
        )

        # EPSG:3857 guarantees square 10x10m pixels globally. 
        # Crucial for CNN spatial filters.
        shared_crs        = "EPSG:3857"
        shared_resolution = 10 

        # ── SENTINEL-1 ────────────────────────────────────────────────────────
        s1_range = (
            f"{(target_dt - pd.to_timedelta('7 days')).strftime('%Y-%m-%d')}/"
            f"{(target_dt + pd.to_timedelta('7 days')).strftime('%Y-%m-%d')}"
        )
        s1_items = list(catalog.search(
            collections=["sentinel-1-rtc"], bbox=bbox, datetime=s1_range
        ).items())

        if not s1_items:
            logging.error(f"[{event_label}] No S1 data in 14-day window. Skipping.")
            return False

        s1_items.sort(key=lambda x: abs(pd.to_datetime(x.datetime) - target_dt))
        best_s1 = s1_items[0]
        s1_delta = abs(pd.to_datetime(best_s1.datetime) - target_dt).days
        logging.info(f"[{event_label}] S1: {best_s1.datetime.date()} (delta={s1_delta}d)")

        assets = best_s1.assets.keys()
        if   'vv' in assets and 'vh' in assets: pol_bands = ['vv', 'vh']
        elif 'hh' in assets and 'hv' in assets: pol_bands = ['hh', 'hv']
        else:
            logging.error(f"[{event_label}] No dual-pol bands found.")
            return False

        s1_frame = odc.stac.load(
            [best_s1], bands=pol_bands, bbox=bbox,
            crs=shared_crs, resolution=shared_resolution,
            patch_url=planetary_computer.sign
        ).isel(time=0)

        # Physical calibration: linear -> dB, clip to [-30, 0] dB, normalize [0, 1]
        s1_valid = s1_frame.where(s1_frame > 0)
        s1_db    = 10 * np.log10(s1_valid)
        s1_norm  = ((s1_db + 30) / 30.0).clip(0, 1)
        s1_norm  = s1_norm.rename({pol_bands[0]: 'SAR_CoPol', pol_bands[1]: 'SAR_CrossPol'})

        # ── SENTINEL-2 ────────────────────────────────────────────────────────
        s2_range = (
            f"{(target_dt - pd.to_timedelta('30 days')).strftime('%Y-%m-%d')}/"
            f"{(target_dt + pd.to_timedelta('14 days')).strftime('%Y-%m-%d')}"
        )
        s2_items = list(catalog.search(
            collections=["sentinel-2-l2a"], bbox=bbox, datetime=s2_range,
            query={"eo:cloud_cover": {"lt": 30}}
        ).items())

        if s2_items:
            s2_items.sort(key=lambda x: abs(pd.to_datetime(x.datetime) - target_dt))
            best_s2    = s2_items[0]
            s2_delta   = abs(pd.to_datetime(best_s2.datetime) - target_dt).days
            s2_cloud   = best_s2.properties.get("eo:cloud_cover", "?")
            logging.info(f"[{event_label}] S2: {best_s2.datetime.date()} "
                         f"(delta={s2_delta}d, cloud={s2_cloud:.1f}%)")

            s2_frame = odc.stac.load(
                [best_s2], bands=["B02", "B03", "B04", "B08", "SCL"],
                bbox=bbox, crs=shared_crs, resolution=shared_resolution,
                patch_url=planetary_computer.sign
            ).isel(time=0)

            opt_bands = s2_frame[["B02", "B03", "B04", "B08"]]
            scl_band  = s2_frame[["SCL"]]

            # Mask nodata (-32768) and normalize reflectance to [0, 1]
            opt_valid = opt_bands.where((opt_bands > 0) & (opt_bands != -32768))
            opt_norm  = (opt_valid / 10000.0).clip(0, 1)

            # Reproject S2 onto S1 grid. 
            # Nearest neighbor for SCL prevents categorical corruption.
            opt_norm = opt_norm.rio.reproject_match(s1_norm)
            scl_band = scl_band.rio.reproject_match(s1_norm, resampling=Resampling.nearest)

            fused_ds = xr.merge([s1_norm, opt_norm, scl_band], compat="override")
            del s2_frame, opt_bands, opt_valid, opt_norm

        else:
            logging.warning(f"[{event_label}] No S2 <30% cloud in 44-day window. Padding NaN.")
            nan_template = xr.full_like(s1_norm['SAR_CoPol'], np.nan)
            fused_ds = xr.Dataset({
                'SAR_CoPol': s1_norm['SAR_CoPol'],
                'SAR_CrossPol': s1_norm['SAR_CrossPol'],
                'B02': nan_template, 'B03': nan_template, 'B04': nan_template, 
                'B08': nan_template, 'SCL': nan_template
            })

        # ── EXPORT ────────────────────────────────────────────────────────────
        # Enforce target band sequence before saving
        fused_ds = fused_ds[BAND_ORDER]
        fused_da = fused_ds.to_array(dim="band").astype("float32")
        fused_da = fused_da.assign_coords(band=BAND_ORDER)
        fused_da.rio.write_nodata(np.nan, inplace=True)
        fused_da.rio.write_crs(shared_crs, inplace=True)

        safe_label = event_label.replace(" ", "_").upper()
        safe_date  = target_date.replace("-", "")
        filename   = f"MANZAR_{safe_label}_S1S2_{safe_date}_{lat:.4f}_{lon:.4f}.tif"
        filepath   = os.path.join(output_dir, filename)

        fused_da.rio.to_raster(filepath, compress="LZW", tiled=True)
        logging.info(f"[{event_label}] SAVED: {filename}")

        del s1_frame, s1_valid, s1_db, s1_norm, fused_ds, fused_da
        gc.collect()
        return True

    except ReadTimeout:
        logging.error(f"[{event_label}] STAC API timeout.")
        return False
    except Exception as e:
        logging.error(f"[{event_label}] FAILED: {str(e)}")
        return False

def run_bulk_download():
    output_dir = "./data/raw_flood_stacks"
    os.makedirs(output_dir, exist_ok=True)

    flood_events = [
        # Pakistan
        {"label": "Sindh_Pakistan_Flood_1", "lat": 26.2, "lon": 68.0, "date": "2022-08-25", "radius": 15},
        {"label": "Sindh_Pakistan_Flood_2", "lat": 27.5, "lon": 68.5, "date": "2022-08-26", "radius": 15},
        {"label": "KPK_Pakistan_Flood", "lat": 34.0, "lon": 71.9, "date": "2022-08-28", "radius": 15},
        {"label": "Balochistan_Pakistan_Flood", "lat": 28.5, "lon": 66.0, "date": "2022-07-25", "radius": 15},
        {"label": "Punjab_Pakistan_Flood", "lat": 30.2, "lon": 70.9, "date": "2023-08-15", "radius": 15},
        
        # South Asia
        {"label": "Assam_India_Flood", "lat": 26.2, "lon": 92.0, "date": "2020-07-15", "radius": 15},
        {"label": "Kerala_India_Flood", "lat": 9.9, "lon": 76.2, "date": "2018-08-16", "radius": 15},
        {"label": "Sylhet_Bangladesh_Flood", "lat": 24.9, "lon": 91.8, "date": "2022-06-18", "radius": 15},
        {"label": "Kathmandu_Nepal_Flood", "lat": 27.7, "lon": 85.3, "date": "2017-08-12", "radius": 10},
        
        # Asia Pacific
        {"label": "Henan_China_Flood", "lat": 34.7, "lon": 113.6, "date": "2021-07-21", "radius": 10},
        {"label": "Jiangxi_China_Flood", "lat": 29.0, "lon": 116.0, "date": "2020-07-12", "radius": 15},
        {"label": "Kyushu_Japan_Flood", "lat": 32.8, "lon": 130.7, "date": "2020-07-04", "radius": 10},
        {"label": "Jakarta_Indonesia_Flood", "lat": -6.2, "lon": 106.8, "date": "2020-01-02", "radius": 10},
        {"label": "New_South_Wales_Australia", "lat": -33.8, "lon": 150.9, "date": "2021-03-22", "radius": 15},
        
        #{"label": "Ahr_Valley_Germany_Flood",  "lat": 50.5,  "lon":   7.0,  "date": "2021-07-15", "radius": 10},
        #{"label": "Emilia_Romagna_Italy",       "lat": 44.5,  "lon":  11.3,  "date": "2023-05-18", "radius": 15},
        #{"label": "Thessaly_Greece_Flood",      "lat": 39.5,  "lon":  22.0,  "date": "2023-09-07", "radius": 15},
        #{"label": "Houston_Texas_Harvey",       "lat": 29.7,  "lon": -95.3,  "date": "2017-08-27", "radius": 15},
        #{"label": "Rio_Grande_Brazil",          "lat": -30.0, "lon": -51.2,  "date": "2024-05-05", "radius": 15},
        #{"label": "British_Columbia_Canada",    "lat": 49.0,  "lon": -122.3, "date": "2021-11-16", "radius": 15},
        #{"label": "Vermont_USA_Flood",          "lat": 44.2,  "lon": -72.5,  "date": "2023-07-11", "radius": 10},
        #{"label": "KwaZulu_Natal_South_Africa", "lat": -29.8, "lon":  31.0,  "date": "2022-04-12", "radius": 10},
        #{"label": "Khartoum_Sudan_Flood",       "lat": 15.6,  "lon":  32.5,  "date": "2020-09-05", "radius": 15},
        #{"label": "Nairobi_Kenya_Flood",        "lat": -1.3,  "lon":  36.8,  "date": "2024-04-25", "radius": 10},
        #{"label": "Beira_Mozambique_Idai",      "lat": -19.8, "lon":  34.8,  "date": "2019-03-15", "radius": 15},
    ]

    logging.info(f"Starting bulk download: {len(flood_events)} events.")
    successful, failed = 0, 0

    for i, event in enumerate(flood_events):
        logging.info(f"--- [{i+1}/{len(flood_events)}] {event['label']} ---")
        max_retries, retry_delay, success = 3, 5, False

        for attempt in range(max_retries):
            success = fetch_fusion_ready_imagery(
                lat=event["lat"], lon=event["lon"], radius_km=event["radius"],
                target_date=event["date"], event_label=event["label"],
                output_dir=output_dir
            )
            if success:
                break
            elif attempt < max_retries - 1:
                logging.warning(f"Retry {attempt+2}/{max_retries} in {retry_delay}s...")
                time.sleep(retry_delay)
                retry_delay *= 2

        successful += success
        failed     += not success
        time.sleep(3)

    logging.info(f"\n=== Done: {successful} success, {failed} failed ===")

if __name__ == "__main__":
    run_bulk_download()

# The Sen1Floods11 Collection Pipeline

In [11]:
import os
import subprocess

def setup_sen1floods11_pipeline(base_dir="./data/sen1floods11"):
    os.makedirs(base_dir, exist_ok=True)
   
    # Verified root path (from your gsutil ls)
    gs_hand_root = "gs://sen1floods11/v1.1/data/flood_events/HandLabeled/"
   
    # CORRECT layers for v1.1 (S1Hand / S2Hand / LabelHand)
    layers = ["S1Hand", "S2Hand", "LabelHand"]
   
    print(f"--- Initiating Sen1Floods11 HandLabeled Download to {base_dir} ---")
   
    for layer in layers:
        local_layer_path = os.path.join(base_dir, layer)
        os.makedirs(local_layer_path, exist_ok=True)
       
        print(f"Fetching Layer: {layer}...")
       
        cmd = f"gsutil -m cp -r -n {gs_hand_root}{layer} {base_dir}"
       
        try:
            subprocess.run(cmd, shell=True, check=True)
            print(f"✅ Successfully synced {layer}.")
        except subprocess.CalledProcessError as e:
            print(f"❌ Failed to sync {layer}. Running verification...")
            subprocess.run(f"gsutil ls {gs_hand_root}", shell=True)
    
    print("\n--- Final Integrity Check ---")
    for layer in layers:
        path = os.path.join(base_dir, layer)
        count = len(os.listdir(path)) if os.path.exists(path) else 0
        print(f"{layer}: {count} chips ready.")

if __name__ == "__main__":
    setup_sen1floods11_pipeline()

--- Initiating Sen1Floods11 HandLabeled Download to ./data/sen1floods11 ---
Fetching Layer: S1Hand...


Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S1Hand/Bolivia_103757_S1Hand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S1Hand/Bolivia_195474_S1Hand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S1Hand/Bolivia_129334_S1Hand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S1Hand/Bolivia_23014_S1Hand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S1Hand/Bolivia_360519_S1Hand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S1Hand/Ghana_103272_S1Hand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S1Hand/Bolivia_312675_S1Hand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S1Hand/Bolivia_432776_S1Hand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S1Hand/Bolivia_242570_S1Hand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S1Hand/Ghana_1033830_S1Hand.tif...
Copying gs://sen1floods11/v1.1/dat

✅ Successfully synced S1Hand.
Fetching Layer: S2Hand...


Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S2Hand/Bolivia_103757_S2Hand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S2Hand/Bolivia_432776_S2Hand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S2Hand/Bolivia_242570_S2Hand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S2Hand/Bolivia_23014_S2Hand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S2Hand/Bolivia_129334_S2Hand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S2Hand/Bolivia_60373_S2Hand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S2Hand/Bolivia_294583_S2Hand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S2Hand/Bolivia_379434_S2Hand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S2Hand/Bolivia_233925_S2Hand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S2Hand/Bolivia_360519_S2Hand.tif...
Copying gs://sen1floods11/v1.1/d

✅ Successfully synced S2Hand.
Fetching Layer: LabelHand...


Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/LabelHand/Bolivia_103757_LabelHand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/LabelHand/Bolivia_129334_LabelHand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/LabelHand/Bolivia_195474_LabelHand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/LabelHand/Bolivia_23014_LabelHand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/LabelHand/Bolivia_233925_LabelHand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/LabelHand/Bolivia_290290_LabelHand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/LabelHand/Bolivia_242570_LabelHand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/LabelHand/Bolivia_294583_LabelHand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/LabelHand/Bolivia_312675_LabelHand.tif...
Copying gs://sen1floods11/v1.1/data/flood_events/HandLabeled/LabelHand/Bol

✅ Successfully synced LabelHand.

--- Final Integrity Check ---
S1Hand: 446 chips ready.
S2Hand: 446 chips ready.
LabelHand: 446 chips ready.


# Visulize the output

In [ ]:
import os
import numpy as np
import rioxarray
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Band index map for 7-band stack
# [0] SAR_CoPol (VV or HH)
# [1] SAR_CrossPol (VH or HV)
# [2] B02 Blue
# [3] B03 Green
# [4] B04 Red
# [5] B08 NIR
# [6] SCL

def percentile_stretch(arr, lo=2, hi=98):
    valid = arr[~np.isnan(arr)]
    if len(valid) == 0:
        return np.zeros_like(arr)
    p_lo, p_hi = np.nanpercentile(valid, lo), np.nanpercentile(valid, hi)
    if p_hi == p_lo:
        return np.zeros_like(arr)
    return np.clip((arr - p_lo) / (p_hi - p_lo), 0, 1)


def make_sar_composite(da):
    vv = da.values[0].astype(np.float32)
    vh = da.values[1].astype(np.float32)
    vv_s = percentile_stretch(vv)
    vh_s = percentile_stretch(vh)
    with np.errstate(divide='ignore', invalid='ignore'):
        ratio = np.where(vh > 0, vv / vh, np.nan)
    ratio_s = percentile_stretch(ratio)
    comp = np.dstack((vv_s, vh_s, ratio_s))
    comp[np.isnan(comp)] = 0
    return comp


def make_rgb(da, r_idx, g_idx, b_idx):
    r = percentile_stretch(da.values[r_idx].astype(np.float32))
    g = percentile_stretch(da.values[g_idx].astype(np.float32))
    b = percentile_stretch(da.values[b_idx].astype(np.float32))
    rgb = np.dstack((r, g, b))
    rgb[np.isnan(rgb)] = 0
    return rgb


def make_ndwi(da):
    green = da.values[3].astype(np.float32)
    nir   = da.values[5].astype(np.float32)
    with np.errstate(divide='ignore', invalid='ignore'):
        ndwi = np.where(
            (~np.isnan(green)) & (~np.isnan(nir)) & (green + nir > 0),
            (green - nir) / (green + nir),
            np.nan
        )
    return ndwi


def check_optical(da):
    """Returns (has_optical, nan_pct). Safe for any band count."""
    if da.shape[0] < 5:
        return False, 100.0
    b04 = da.values[4].astype(np.float32)
    nan_pct = np.sum(np.isnan(b04)) / b04.size * 100
    return nan_pct < 70.0, nan_pct


def visualize_flood_stack(filepath):
    if not os.path.exists(filepath):
        print(f"File not found: {filepath}")
        return

    da = rioxarray.open_rasterio(filepath)
    fname = os.path.basename(filepath)
    n_bands = da.shape[0]

    print(f"File    : {fname}")
    print(f"Shape   : {da.values.shape}  (bands, height, width)")
    print(f"CRS     : {da.rio.crs}")
    print(f"NoData  : {da.rio.nodata}")
    print(f"Bands   : {n_bands}")

    has_sar     = n_bands >= 2
    optical_ok, nan_pct = check_optical(da)

    if not has_sar:
        print("ERROR: Need at least 2 SAR bands. Skipping.")
        da.close()
        return

    if not optical_ok:
        print(f"WARNING: Optical NaN={nan_pct:.1f}% — SAR panels only.")

    n_cols = 4 if optical_ok else 2
    fig, axes = plt.subplots(1, n_cols, figsize=(7 * n_cols, 7),
                             gridspec_kw={'wspace': 0.05})
    if n_cols == 1:
        axes = [axes]

    fig.suptitle(fname, fontsize=10, fontweight='bold')

    col = 0

    # SAR grayscale (VV/HH)
    vv_s = percentile_stretch(da.values[0].astype(np.float32))
    vv_s[np.isnan(vv_s)] = 0
    axes[col].imshow(vv_s, cmap='gray', interpolation='nearest')
    axes[col].set_title("SAR CoPol\n(Grayscale)", fontsize=11, fontweight='bold')
    axes[col].axis('off')
    col += 1

    # SAR false color composite
    sar_comp = make_sar_composite(da)
    axes[col].imshow(sar_comp, interpolation='nearest')
    axes[col].set_title("SAR False Color\n(R=VV  G=VH  B=VV/VH)", fontsize=11, fontweight='bold')
    axes[col].axis('off')
    col += 1

    if optical_ok:
        # S2 true color
        rgb = make_rgb(da, r_idx=4, g_idx=3, b_idx=2)
        axes[col].imshow(rgb, interpolation='nearest')
        axes[col].set_title("S2 True Color\n(R=B04  G=B03  B=B02)", fontsize=11, fontweight='bold')
        axes[col].axis('off')
        col += 1

        # NDWI
        ndwi = make_ndwi(da)
        im = axes[col].imshow(ndwi, cmap='RdYlBu', vmin=-0.5, vmax=0.5,
                              interpolation='nearest')
        plt.colorbar(im, ax=axes[col], fraction=0.03, pad=0.02)
        axes[col].set_title("NDWI\n(Blue = water)", fontsize=11, fontweight='bold')
        axes[col].axis('off')

    plt.show()
    plt.close()
    da.close()


def visualize_all(directory):
    from pathlib import Path
    tif_files = sorted(Path(directory).glob("*.tif"))
    if not tif_files:
        print(f"No .tif files in {directory}")
        return
    print(f"Visualizing {len(tif_files)} files...\n")
    for fp in tif_files:
        visualize_flood_stack(str(fp))
        print()



# Single file
#visualize_flood_stack(
#    "/kaggle/working/data/raw_flood_stacks/MANZAR_RIO_GRANDE_BRAZIL_S1S2_20240505_-30.0000_-51.2000.tif"
#)

# Or all files
visualize_all("/kaggle/working/data/raw_flood_stacks")

# Quality Checker for all images

In [14]:
import os
import rioxarray
import numpy as np
import pandas as pd
from pathlib import Path

# Canonical band order — must match download script's BAND_ORDER
# [0] SAR_CoPol  [1] SAR_CrossPol  [2] B02  [3] B03  [4] B04  [5] B08  [6] SCL
BAND_NAMES = [
    "SAR_CoPol",
    "SAR_CrossPol",
    "B02 (Blue)",
    "B03 (Green)",
    "B04 (Red)",
    "B08 (NIR)",
    "SCL (Mask)",
]

# SCL integer class definitions (ESA standard)
SCL_CLASSES = {
    0:  "No Data",
    1:  "Saturated/Defective",
    2:  "Dark Area",
    3:  "Cloud Shadow",
    4:  "Vegetation",
    5:  "Bare Soil",
    6:  "Water",
    7:  "Unclassified",
    8:  "Cloud Medium Prob",
    9:  "Cloud High Prob",
    10: "Thin Cirrus",
    11: "Snow/Ice",
}


def _band_stats(arr, total_pixels):
    """Compute per-band stats. Returns dict."""
    data = arr.astype(np.float64)
    nans = np.isnan(data)
    infs = np.isinf(data)
    valid = data[~nans & ~infs]
    nan_pct  = np.sum(nans) / total_pixels * 100
    inf_pct  = np.sum(infs) / total_pixels * 100
    zero_pct = np.sum(data == 0) / total_pixels * 100

    if len(valid) == 0:
        return {
            "Min": np.nan, "Max": np.nan, "Mean": np.nan, "Std": np.nan,
            "NaN%": nan_pct, "Inf%": inf_pct, "Zero%": zero_pct
        }
    return {
        "Min":   float(np.min(valid)),
        "Max":   float(np.max(valid)),
        "Mean":  float(np.mean(valid)),
        "Std":   float(np.std(valid)),
        "NaN%":  nan_pct,
        "Inf%":  inf_pct,
        "Zero%": zero_pct,
    }


def _parse_scl(scl_arr, total_pixels):
    """
    Parse SCL band into per-class percentages.
    Detects if SCL was accidentally normalized to [0,1] and recovers integer classes.
    Returns (class_pct dict, was_normalized bool).
    """
    data   = scl_arr.astype(np.float64)
    valid  = data[~np.isnan(data)]

    if len(valid) == 0:
        return {}, False, "SCL entirely NaN"

    max_val = float(np.nanmax(valid))

    # If max <= 1.0 and values aren't whole numbers → SCL was normalized
    was_normalized = (max_val <= 1.0) and not np.all(valid == np.round(valid))
    warning = None

    if was_normalized:
        # Recover: undo /10000 normalization
        valid = np.round(valid * 10000).astype(int)
        warning = "SCL_NORMALIZED: SCL was divided by 10000 — old tile. Re-download recommended."
    else:
        valid = np.round(valid).astype(int)
        # Clamp to valid SCL range
        valid = valid[(valid >= 0) & (valid <= 11)]

    unique, counts = np.unique(valid, return_counts=True)
    class_pct = {int(k): float(v / total_pixels * 100) for k, v in zip(unique, counts)}
    return class_pct, was_normalized, warning


def dl_preflight_qa(filepath):
    try:
        da = rioxarray.open_rasterio(filepath)
    except Exception as e:
        return None, None, [f"CRITICAL: Failed to load — {e}"]

    n_bands, height, width = da.shape
    total_pixels = height * width

    if n_bands != len(BAND_NAMES):
        da.close()
        return None, None, [f"CRITICAL: Expected {len(BAND_NAMES)} bands, found {n_bands}."]

    # ── Per-band statistics ────────────────────────────────────────────────
    rows = []
    stats_by_band = {}
    for i, name in enumerate(BAND_NAMES):
        s = _band_stats(da.values[i], total_pixels)
        stats_by_band[name] = s
        rows.append({
            "Band":   name,
            "Min":    f"{s['Min']:.4f}"  if not np.isnan(s['Min'])  else "NaN",
            "Max":    f"{s['Max']:.4f}"  if not np.isnan(s['Max'])  else "NaN",
            "Mean":   f"{s['Mean']:.4f}" if not np.isnan(s['Mean']) else "NaN",
            "Std":    f"{s['Std']:.4f}"  if not np.isnan(s['Std'])  else "NaN",
            "NaN%":   f"{s['NaN%']:.2f}%",
            "Inf%":   f"{s['Inf%']:.2f}%",
            "Zero%":  f"{s['Zero%']:.2f}%",
        })
    df = pd.DataFrame(rows)

    # ── SCL analysis ───────────────────────────────────────────────────────
    class_pct, scl_normalized, scl_warning = _parse_scl(da.values[6], total_pixels)

    cloud_pct  = class_pct.get(8, 0.0) + class_pct.get(9, 0.0) + class_pct.get(10, 0.0)
    shadow_pct = class_pct.get(3, 0.0)
    water_pct  = class_pct.get(6, 0.0)
    nodata_pct = class_pct.get(0, 0.0)

    scl_summary = {
        "cloud":  cloud_pct,
        "shadow": shadow_pct,
        "water":  water_pct,
        "nodata": nodata_pct,
        "classes": class_pct,
    }

    # ── Flag logic ─────────────────────────────────────────────────────────
    flags = []

    # SCL corruption
    if scl_warning:
        flags.append(scl_warning)

    # Band-level NaN — check all bands, threshold at 50%
    for name in BAND_NAMES:
        nan_pct = stats_by_band[name]["NaN%"]
        if nan_pct > 50:
            flags.append(f"HIGH_NAN [{name}]: {nan_pct:.1f}%")
        elif nan_pct > 20:
            flags.append(f"PARTIAL_NAN [{name}]: {nan_pct:.1f}%")

    # SAR-specific: check for zero-dominated bands (nodata written as 0, not NaN)
    for sar_band in ["SAR_CoPol", "SAR_CrossPol"]:
        zero_pct = stats_by_band[sar_band]["Zero%"]
        if zero_pct > 50:
            flags.append(f"SAR_ZERO_DOMINATED [{sar_band}]: {zero_pct:.1f}% zeros — nodata not NaN")

    # SAR value range sanity (after dB normalization to [0,1])
    for sar_band in ["SAR_CoPol", "SAR_CrossPol"]:
        s = stats_by_band[sar_band]
        if not np.isnan(s["Max"]):
            if s["Max"] > 10:
                flags.append(f"SAR_NOT_NORMALIZED [{sar_band}]: max={s['Max']:.1f} — still in raw dB or linear scale")
            if s["Min"] < 0:
                flags.append(f"SAR_NEGATIVE [{sar_band}]: min={s['Min']:.4f} — unexpected")

    # Optical value range sanity (after /10000 normalization to [0,1])
    for opt_band in ["B02 (Blue)", "B03 (Green)", "B04 (Red)", "B08 (NIR)"]:
        s = stats_by_band[opt_band]
        if not np.isnan(s["Min"]) and s["Min"] < 0:
            flags.append(f"OPTICAL_NEGATIVE [{opt_band}]: min={s['Min']:.4f} — nodata bleed (-32768 not masked)")
        if not np.isnan(s["Max"]) and s["Max"] > 1.01:
            flags.append(f"OPTICAL_OVERFLOW [{opt_band}]: max={s['Max']:.4f} — not normalized to [0,1]")

    # Scale mismatch between optical and SAR (catches old un-normalized tiles)
    nir_max = stats_by_band["B08 (NIR)"]["Max"]
    vv_max  = stats_by_band["SAR_CoPol"]["Max"]
    if not np.isnan(nir_max) and not np.isnan(vv_max):
        if nir_max > 100 and vv_max < 10:
            flags.append(f"SCALE_MISMATCH: optical_max={nir_max:.0f} vs SAR_max={vv_max:.2f}")

    # SCL-based flags
    if water_pct == 0 and not scl_normalized:
        flags.append("NO_WATER_IN_SCL: SCL class 6 absent — flood signal may not be visible in optical")
    if cloud_pct > 30:
        flags.append(f"HIGH_CLOUD: {cloud_pct:.1f}% — optical bands likely corrupted")
    if shadow_pct > 20:
        flags.append(f"HIGH_SHADOW: {shadow_pct:.1f}%")
    if nodata_pct > 30:
        flags.append(f"HIGH_SCL_NODATA: {nodata_pct:.1f}% — tile boundary or missing acquisition")

    # Footprint mismatch: optical and SAR NaN% differ significantly
    sar_nan  = stats_by_band["SAR_CoPol"]["NaN%"]
    opt_nan  = stats_by_band["B04 (Red)"]["NaN%"]
    if abs(sar_nan - opt_nan) > 5 and not (np.isnan(sar_nan) or np.isnan(opt_nan)):
        flags.append(
            f"FOOTPRINT_MISMATCH: SAR_NaN={sar_nan:.1f}% vs B04_NaN={opt_nan:.1f}% "
            f"— S1/S2 grids not aligned"
        )

    da.close()
    return df, scl_summary, flags


def run_batch_qa(directory):
    tif_files = sorted(Path(directory).glob("*.tif"))

    if not tif_files:
        print(f"No .tif files found in {directory}")
        return

    print(f"BATCH QA  |  {len(tif_files)} files  |  {directory}\n")

    clean, flagged, failed = [], [], []

    for idx, filepath in enumerate(tif_files, 1):
        fname = filepath.name
        print(f"{'='*72}")
        print(f"[{idx}/{len(tif_files)}] {fname}")
        print(f"{'='*72}")

        band_df, scl, flags = dl_preflight_qa(filepath)

        if band_df is None:
            print(f"  ERROR: {flags[0]}")
            failed.append(fname)
            print()
            continue

        # Band stats table
        print(band_df.to_string(index=False))

        # SCL summary
        print(f"\nSCL Summary:")
        print(f"  Cloud (8+9+10): {scl['cloud']:.2f}%  |  "
              f"Shadow (3): {scl['shadow']:.2f}%  |  "
              f"Water (6): {scl['water']:.2f}%  |  "
              f"NoData (0): {scl['nodata']:.2f}%")

        # Non-zero SCL classes
        active = {SCL_CLASSES.get(k, f"Class {k}"): f"{v:.2f}%"
                  for k, v in sorted(scl['classes'].items()) if v > 0.5}
        if active:
            print(f"  Active classes: {active}")

        # Flags
        if flags:
            print(f"\nRED FLAGS ({len(flags)}):")
            for f in flags:
                print(f"  -> {f}")
            flagged.append(fname)
        else:
            print(f"\nSTATUS: CLEAN")
            clean.append(fname)

        print()

    # Summary
    print(f"{'='*72}")
    print(f"DONE  |  Total: {len(tif_files)}  |  "
          f"Clean: {len(clean)}  |  Flagged: {len(flagged)}  |  Failed: {len(failed)}")

    if clean:
        print(f"\nCLEAN ({len(clean)}):")
        for f in clean:
            print(f"  {f}")
    if flagged:
        print(f"\nFLAGGED ({len(flagged)}):")
        for f in flagged:
            print(f"  {f}")
    if failed:
        print(f"\nFAILED ({len(failed)}):")
        for f in failed:
            print(f"  {f}")


run_batch_qa("/kaggle/working/data/raw_flood_stacks")

BATCH QA  |  25 files  |  /kaggle/working/data/raw_flood_stacks

[1/25] MANZAR_AHR_VALLEY_GERMANY_FLOOD_S1S2_20210715_50.5000_7.0000.tif
        Band    Min    Max   Mean    Std   NaN%  Inf% Zero%
   SAR_CoPol 0.1404 1.0000 0.7550 0.0961  0.00% 0.00% 0.00%
SAR_CrossPol 0.0047 1.0000 0.5618 0.0894  0.00% 0.00% 0.00%
  B02 (Blue) 0.0048 1.0000 0.1037 0.1442 20.32% 0.00% 0.00%
 B03 (Green) 0.0130 1.0000 0.1210 0.1381 20.32% 0.00% 0.00%
   B04 (Red) 0.0093 1.0000 0.1069 0.1432 20.32% 0.00% 0.00%
   B08 (NIR) 0.0329 1.0000 0.3844 0.1425 20.32% 0.00% 0.00%
  SCL (Mask) 2.0000 9.0000 4.7542 1.6348 20.32% 0.00% 0.00%

SCL Summary:
  Cloud (8+9+10): 10.11%  |  Shadow (3): 1.59%  |  Water (6): 0.00%  |  NoData (0): 0.00%
  Active classes: {'Cloud Shadow': '1.59%', 'Vegetation': '59.42%', 'Bare Soil': '4.96%', 'Unclassified': '3.39%', 'Cloud Medium Prob': '3.59%', 'Cloud High Prob': '6.52%'}

RED FLAGS (7):
  -> PARTIAL_NAN [B02 (Blue)]: 20.3%
  -> PARTIAL_NAN [B03 (Green)]: 20.3%
  -> PARTIAL_NA

# Calculate NDWI/SAR indices and slice massive regions into ML-ready 512x512 tiles.

In [1]:
import os
import glob
import numpy as np
import rasterio
import pandas as pd
from tqdm import tqdm

def build_sen1floods_tensors(base_dir="./data/sen1floods11", output_dir="./data/ml_dataset"):
    """
    Phase 2: Ingests Sen1Floods11 Triplets, engineers features, 
    and outputs standardized 8-channel PyTorch tensors.
    """
    s1_dir = os.path.join(base_dir, 'S1Hand')
    s2_dir = os.path.join(base_dir, 'S2Hand')
    lbl_dir = os.path.join(base_dir, 'LabelHand')

    feat_dir = os.path.join(output_dir, 'features')
    mask_dir = os.path.join(output_dir, 'masks')
    os.makedirs(feat_dir, exist_ok=True)
    os.makedirs(mask_dir, exist_ok=True)

    s1_files = glob.glob(os.path.join(s1_dir, '*.tif'))
    if not s1_files:
        print(f"Error: No S1 files found in {s1_dir}")
        return

    manifest = []
    
    print(f"Engineering features and stacking {len(s1_files)} triplets...")

    for s1_path in tqdm(s1_files, desc="Processing Tensors"):
        filename = os.path.basename(s1_path)
        base_name = filename.replace('_S1Hand.tif', '')

        s2_path = os.path.join(s2_dir, f"{base_name}_S2Hand.tif")
        lbl_path = os.path.join(lbl_dir, f"{base_name}_LabelHand.tif")

        # Skip if the triplet is incomplete
        if not (os.path.exists(s2_path) and os.path.exists(lbl_path)):
            continue

        # Load Data
        with rasterio.open(s1_path) as src:
            s1_data = src.read() # Shape: [2, 512, 512]
        with rasterio.open(s2_path) as src:
            s2_data = src.read() # Shape: [13, 512, 512]
        with rasterio.open(lbl_path) as src:
            lbl_data = src.read(1) # Shape: [512, 512]

        # --- 1. SAR CALIBRATION ---
        # Convert linear backscatter to Decibels and scale to [0, 1]
        s1_valid = np.where(s1_data > 0, s1_data, 1e-7)
        s1_db = 10 * np.log10(s1_valid)
        s1_norm = np.clip((s1_db + 30) / 30.0, 0, 1)
        
        copol, crosspol = s1_norm[0], s1_norm[1]

        # --- 2. OPTICAL NORMALIZATION ---
        # Sen1Floods11 Optical Bands: 1=Blue, 2=Green, 3=Red, 7=NIR
        blue = np.clip(s2_data[1] / 10000.0, 0, 1)
        green = np.clip(s2_data[2] / 10000.0, 0, 1)
        red = np.clip(s2_data[3] / 10000.0, 0, 1)
        nir = np.clip(s2_data[7] / 10000.0, 0, 1)

        # --- 3. ENGINEERED INDICES ---
        with np.errstate(divide='ignore', invalid='ignore'):
            # NDWI = (Green - NIR) / (Green + NIR)
            ndwi = np.where((green + nir) > 0, (green - nir) / (green + nir), 0.0)
            
            # SAR Ratio = VV / VH
            sar_ratio = np.where(crosspol > 0, copol / crosspol, 0.0)

        # Sanitize math artifacts
        ndwi = np.nan_to_num(ndwi, nan=0.0, posinf=1.0, neginf=-1.0)
        sar_ratio = np.nan_to_num(sar_ratio, nan=0.0, posinf=0.0, neginf=0.0)

        # --- 4. TENSOR STACKING ---
        # Enforce strict 8-channel layout
        features = np.stack([copol, crosspol, blue, green, red, nir, ndwi, sar_ratio], axis=0).astype(np.float32)
        mask = lbl_data.astype(np.int8)

        # --- 5. EXPORT ---
        patch_id = f"SEN1_{base_name}"
        np.save(os.path.join(feat_dir, f"{patch_id}.npy"), features)
        np.save(os.path.join(mask_dir, f"{patch_id}.npy"), mask)

        # --- 6. MANIFEST METRICS ---
        # Calculate water ratio ignoring the -1 cloud pixels
        valid_pixels = np.sum(mask != -1)
        water_pixels = np.sum(mask == 1)
        water_ratio = (water_pixels / valid_pixels) if valid_pixels > 0 else 0.0

        manifest.append({
            "patch_id": patch_id,
            "biome": base_name.split('_')[0], # Captures region (e.g., 'Bolivia', 'Pakistan')
            "water_ratio": round(water_ratio, 4),
            "valid_pixel_ratio": round(valid_pixels / (512 * 512), 4)
        })

    # Save tracking manifest
    df = pd.DataFrame(manifest)
    df.to_csv(os.path.join(output_dir, 'patch_manifest.csv'), index=False)
    
    print("\n=== Phase 2 Complete ===")
    print(f"Successfully serialized {len(df)} 8-channel tensors.")
    print(f"Patches containing water: {len(df[df['water_ratio'] > 0.01])}")
    print(f"Dataset footprint saved to: {output_dir}")

# Execution
if __name__ == "__main__":
    build_sen1floods_tensors()

Engineering features and stacking 446 triplets...


Processing Tensors: 100%|██████████| 446/446 [00:33<00:00, 13.17it/s]


=== Phase 2 Complete ===
Successfully serialized 446 8-channel tensors.
Patches containing water: 293
Dataset footprint saved to: ./data/ml_dataset


# Architecture & Training

In [ ]:
import os
import gc
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms.functional as TF
import segmentation_models_pytorch as smp
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# --- CONFIGURATION ---
class Config:
    seed = 42
    manifest_path = "./data/ml_dataset/patch_manifest.csv"
    features_dir = "./data/ml_dataset/features"
    masks_dir = "./data/ml_dataset/masks"
    checkpoint_dir = "./checkpoints"
    
    # Hyperparams
    lr = 1e-4
    weight_decay = 1e-4
    batch_size = 8
    accumulation_steps = 4
    epochs = 150
    patience = 12
    val_size = 0.2
    
    # Hardware
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    num_workers = 4

def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

# --- DATASET & SPLITTING ---
class FloodDataset(Dataset):
    def __init__(self, df, features_dir, masks_dir, is_train=True):
        self.df = df
        self.features_dir = features_dir
        self.masks_dir = masks_dir
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        patch_id = self.df.iloc[idx]['patch_id']
        x = np.load(os.path.join(self.features_dir, f"{patch_id}.npy"))
        y = np.load(os.path.join(self.masks_dir, f"{patch_id}.npy"))

        x = torch.from_numpy(x).float()
        y = torch.from_numpy(y).long()

        if self.is_train:
            # Geometric Augmentations
            if random.random() > 0.5:
                x = TF.hflip(x); y = TF.hflip(y)
            if random.random() > 0.5:
                x = TF.vflip(x); y = TF.vflip(y)
            
            k = random.choice([0, 1, 2, 3])
            if k > 0:
                x = torch.rot90(x, k, [1, 2]); y = torch.rot90(y, k, [0, 1])

            # Modality Dropout (Partial)
            if random.random() < 0.15: # Randomly drop optical channels (2:7)
                x[2:7, :, :] = 0.0

        return x, y

def get_dataloaders(config):
    df = pd.read_csv(config.manifest_path)
    
    # Stratified split based on water density
    df['strata'] = pd.qcut(df['water_ratio'], q=5, labels=False, duplicates='drop')
    train_df, val_df = train_test_split(
        df, test_size=config.val_size, stratify=df['strata'], random_state=config.seed
    )
    
    # Weighted Sampler for Training (addressing imbalance)
    weights = np.where(train_df['water_ratio'] > 0.01, 3.0, 1.0)
    sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

    train_ds = FloodDataset(train_df, config.features_dir, config.masks_dir, is_train=True)
    val_ds = FloodDataset(val_df, config.features_dir, config.masks_dir, is_train=False)

    train_loader = DataLoader(train_ds, batch_size=config.batch_size, sampler=sampler, 
                              num_workers=config.num_workers, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=config.batch_size, shuffle=False, 
                            num_workers=config.num_workers, pin_memory=True)
    
    return train_loader, val_loader

# --- ARCHITECTURE ---
class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=8):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(in_planes, in_planes // ratio, 1, bias=False),
            nn.ReLU(),
            nn.Conv2d(in_planes // ratio, in_planes, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        return x * self.sigmoid(avg_out + max_out)

class ProductionDualStreamUnet(nn.Module):
    def __init__(self):
        super().__init__()
        # Use UnetPlusPlus for deeper skip connections
        self.sar_encoder = smp.UnetPlusPlus(encoder_name="resnet34", in_channels=3, classes=64)
        self.opt_encoder = smp.UnetPlusPlus(encoder_name="resnet34", in_channels=5, classes=64)
        
        self.fusion = nn.Sequential(
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            ChannelAttention(64),
            nn.Conv2d(64, 1, kernel_size=1)
        )

    def forward(self, x):
        # SAR: VV, VH, VV/VH ratio
        sar_feat = self.sar_encoder(torch.cat([x[:,:2,:,:], x[:,7:8,:,:]], dim=1))
        # Optical: Multi-band
        opt_feat = self.opt_encoder(x[:,2:7,:,:])
        
        fused = torch.cat([sar_feat, opt_feat], dim=1)
        return self.fusion(fused)

# --- METRIC TRACKING ---
class MetricTracker:
    def __init__(self):
        self.reset()

    def reset(self):
        self.tp, self.fp, self.fn, self.tn = 0, 0, 0, 0

    def update(self, output, target):
        preds = (torch.sigmoid(output) > 0.5).long().view(-1)
        target = target.view(-1)
        
        # Mask out invalid labels (assuming -1 is ignore)
        mask = target != -1
        preds = preds[mask]
        target = target[mask]

        self.tp += ((preds == 1) & (target == 1)).sum().item()
        self.fp += ((preds == 1) & (target == 0)).sum().item()
        self.fn += ((preds == 0) & (target == 1)).sum().item()
        self.tn += ((preds == 0) & (target == 0)).sum().item()

    def compute(self):
        eps = 1e-7
        precision = self.tp / (self.tp + self.fp + eps)
        recall = self.tp / (self.tp + self.fn + eps)
        f1 = 2 * (precision * recall) / (precision + recall + eps)
        iou = self.tp / (self.tp + self.fp + self.fn + eps)
        return {"precision": precision, "recall": recall, "f1": f1, "iou": iou}

# --- ENGINE ---
class Trainer:
    def __init__(self, config):
        self.config = config
        self.model = ProductionDualStreamUnet().to(config.device)
        self.optimizer = optim.AdamW(self.model.parameters(), lr=config.lr, weight_decay=config.weight_decay)
        self.scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(self.optimizer, T_0=10, T_mult=2)
        self.criterion = smp.losses.DiceLoss(mode='binary', from_logits=True)
        self.scaler = torch.amp.GradScaler('cuda')
        self.tracker = MetricTracker()
        
        os.makedirs(config.checkpoint_dir, exist_ok=True)

    def train_epoch(self, loader):
        self.model.train()
        self.tracker.reset()
        running_loss = 0
        
        pbar = tqdm(loader, desc="Training", leave=False)
        for i, (x, y) in enumerate(pbar):
            x, y = x.to(self.config.device), y.to(self.config.device).float().unsqueeze(1)
            
            with torch.amp.autocast('cuda'):
                logits = self.model(x)
                loss = self.criterion(logits, y) / self.config.accumulation_steps
            
            self.scaler.scale(loss).backward()
            
            if (i + 1) % self.config.accumulation_steps == 0:
                self.scaler.step(self.optimizer)
                self.scaler.update()
                self.optimizer.zero_grad()
            
            running_loss += loss.item() * self.config.accumulation_steps
            self.tracker.update(logits.detach(), y)
            pbar.set_postfix(loss=f"{loss.item():.4f}")
            
        return running_loss / len(loader), self.tracker.compute()

    @torch.no_grad()
    def validate(self, loader):
        self.model.eval()
        self.tracker.reset()
        running_loss = 0
        
        for x, y in loader:
            x, y = x.to(self.config.device), y.to(self.config.device).float().unsqueeze(1)
            logits = self.model(x)
            loss = self.criterion(logits, y)
            running_loss += loss.item()
            self.tracker.update(logits, y)
            
        return running_loss / len(loader), self.tracker.compute()

    def save_checkpoint(self, epoch, iou, is_best=False):
        state = {
            'epoch': epoch,
            'state_dict': self.model.state_dict(),
            'optimizer': self.optimizer.state_dict(),
            'scheduler': self.scheduler.state_dict(),
            'best_iou': iou
        }
        name = "best_model.pth" if is_best else "last_model_v3.pth"
        torch.save(state, os.path.join(self.config.checkpoint_dir, name))

def main():
    config = Config()
    seed_everything(config.seed)
    train_loader, val_loader = get_dataloaders(config)
    trainer = Trainer(config)
    
    best_iou = 0
    patience_counter = 0
    
    print(f"Starting Training on {config.device}...")
    print(f"{'Epoch':<5} | {'T-Loss':<7} | {'V-Loss':<7} | {'V-IoU':<7} | {'V-F1':<7}")
    
    for epoch in range(config.epochs):
        t_loss, t_metrics = trainer.train_epoch(train_loader)
        v_loss, v_metrics = trainer.validate(val_loader)
        
        trainer.scheduler.step()
        
        print(f"{epoch+1:5d} | {t_loss:.4f} | {v_loss:.4f} | {v_metrics['iou']:.4f} | {v_metrics['f1']:.4f}")
        
        # Checkpointing Logic
        if v_metrics['iou'] > best_iou:
            best_iou = v_metrics['iou']
            trainer.save_checkpoint(epoch, best_iou, is_best=True)
            patience_counter = 0
        else:
            patience_counter += 1
            trainer.save_checkpoint(epoch, best_iou, is_best=False)

        if patience_counter >= config.patience:
            print("Early Stopping Triggered.")
            break

if __name__ == "__main__":
    main()

Starting Training on cuda...
Epoch | T-Loss  | V-Loss  | V-IoU   | V-F1   


    1 | 0.5893 | 0.2967 | 0.2133 | 0.3516


    2 | 0.4325 | 0.2765 | 0.2855 | 0.4441


    3 | 0.3366 | 0.2666 | 0.3494 | 0.5179


    4 | 0.4558 | 0.2648 | 0.3266 | 0.4924


...

   11 | 0.4557 | 0.2580 | 0.4425 | 0.6135


   12 | 0.2868 | 0.2565 | 0.3646 | 0.5344


   13 | 0.3274 | 0.2556 | 0.3493 | 0.5177


   14 | 0.4049 | 0.2594 | 0.3617 | 0.5313


Training:  64%|██████▍   | 29/45 [00:36<00:19,  1.24s/it, loss=0.0000]

# Delete files

In [8]:
#!rm -rf /kaggle/working/manzar_dual_stream_epoch_3.pth
#import os, glob
#for f in glob.glob("/kaggle/working/data/raw_flood_stacks/*.tif"):
#    os.remove(f)
print("Cleared.")

Cleared.
